# 04 · Claude Code RFT: coding episodes inside Claude Code (generation)

The RFT-Agent model solves coding tasks **inside Claude Code**, with the same setup as the eval in `03`: the Q4 GGUF
on llama-server, the proxy of `eval/claude_code.py`, `claude -p` with the tools used at home
(`Bash, Read, Edit, Write, Glob, Grep` and the web tools of `eval/web_mcp.py`). The model learns to work in Claude
Code from its own successful runs.

**Tasks:**
- KodCode (one function in `solution.py`, given tests).
- **Repository tasks** in 7 small Python libraries (`CC.TRAIN_REPOS`: toolz, cachetools, boltons, more-itertools,
  click, mistune, jmespath): a bug put into the source, or a function body removed, so that some tests fail. The
  model has to find the place in the repository, which is what Claude Code is used for. The libraries of the
  `cc_repo` eval in `03` are different ones.

**How it runs:**
- The proxy keeps the last request of every run: the whole conversation (Claude Code's system prompt and tools, the
  task, every reply with its thinking, every tool result) in OpenAI format, ready for the chat template.
- Every run is graded by the task's tests after the test file is restored.
- `05` trains on the selected episodes (`claude_code_sft.jsonl`) together with the data of `01` and `02`.

**Selection (same rules as 02):**
- Only runs that ended on their own (`success`), passed the restored tests and fit the training length of
  **16384 tokens** (`05`); shortest first. Up to 2 per partially solved task, one from half of the fully solved tasks.
- **Clean episodes only:** no user message after the model's first reply. The chat template drops the thinking of
  every reply before the last user message, so such a reply would be trained without its thinking.
- **Whole conversations only:** Claude Code runs without auto-compaction (`compact=False`). A compacted run keeps
  only the summary and what came after it; a run that outgrows the context stops ("Prompt is too long") and is
  not kept. The training length (16384) is well below what fits.

**Runtime and cost:**
- Runtime: **A100 80GB** (≈6.8 CU/hour).
- Setup ~25 min (with the repository tasks), generation at most `MAX_GEN_HOURS`. **~30–35 CU** in total.
- Resumable: results and conversations are appended to Drive. After a disconnect run the notebook again.

**Before you start:** copy the repo's current `eval/` folder to Drive. `01` and `02` should have run, so that their
problems are left out of the pool.

## Keep the browser tab alive (phone)

Colab drops the session when the browser freezes the tab, which happens when the phone screen turns off.
A tab that is playing audio is not frozen, so the cell below plays a very quiet tone in a loop. Run it first.
- If nothing plays, tap ▶ on the player. Chrome's media notification should appear.
- Turn the media volume down, but not to zero. Another app playing audio (e.g. Spotify) pauses it.
- Android (Xiaomi/HyperOS): set Chrome's battery saver to *No restrictions* and lock Chrome in recent apps.
- The player keeps playing after *Restart session*; if it stops, run the cell again.

Long steps still write to Drive and resume when rerun, so a dropped session only loses the step in progress.

In [ ]:
# Plays a very quiet 200 Hz tone in a loop so the browser does not freeze this tab
import numpy as np
from IPython.display import Audio, HTML, display

_sr = 8000
_wave = 0.01 * np.sin(2 * np.pi * 200 * np.arange(_sr * 5) / _sr)
display(HTML(Audio(_wave, rate=_sr, autoplay=True, normalize=False)._repr_html_().replace("<audio ", "<audio loop ")))

In [ ]:
# Node.js and Claude Code, pinned: every eval and the training data use the same Claude Code
import os
NODE_VERSION, CLAUDE_CODE_VERSION = "22.12.0", "2.1.331"
NODE_BIN = f"/opt/node-v{NODE_VERSION}-linux-x64/bin"
CLAUDE_BIN = f"{NODE_BIN}/claude"
!pip install -q pytest datasets requests "ddgs==9.16.1" simplejson wcwidth typing_extensions   # web tools; test deps of the repositories
!curl -fsSL https://nodejs.org/dist/v{NODE_VERSION}/node-v{NODE_VERSION}-linux-x64.tar.xz | tar -xJ -C /opt
os.environ["PATH"] = f"{NODE_BIN}:{os.environ['PATH']}"
!npm install -g --silent @anthropic-ai/claude-code@{CLAUDE_CODE_VERSION}
!{CLAUDE_BIN} --version
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# llama.cpp server, CUDA build for this GPU only (~10 min)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null
!cmake --build /content/llama.cpp/build -j --target llama-server 2>&1 | tail -3
LLAMA_SERVER = "/content/llama.cpp/build/bin/llama-server"

## Settings

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/colab-ai"
EVAL_DIR = f"{DRIVE_ROOT}/eval"
ROUND1_GEN = f"{DRIVE_ROOT}/rft/round1/generations.jsonl"
AGENT1_EPISODES = f"{DRIVE_ROOT}/rft/agent1/episodes.jsonl"
HARNESS1_TASKS = f"{DRIVE_ROOT}/rft/harness1/tasks.jsonl"     # v0.1.0, if it exists
OUT_DIR = f"{DRIVE_ROOT}/rft/claude_code2"   # claude_code1: every run was compacted (unusable)
WORK = "/srv/cc_work"

GGUF_REPO, GGUF_PATTERN = "salihyesil59/Qwen3.6-35B-A3B-RFT-Agent-GGUF", "Q4_K_M"   # the best model so far
TOKENIZER = "Qwen/Qwen3.6-35B-A3B-FP8"     # chat template and tokenizer for the selection (as in 05)

# Generation
POOL = 250                  # KodCode tasks (medium/hard), disjoint from every eval set and earlier rounds
REPO_TASKS_PER_REPO = 22    # repository tasks: 7 libraries, ~150 tasks
SAMPLES = 2                 # runs per task
SLOTS = 24                  # parallel conversations on the server = runs at the same time
SLOT_CONTEXT = 32768
TURN_TOKENS = 8192          # leaves 32768 - 8192 - 3000 tokens for the conversation (no compaction, see the top)
MAX_TURNS = 40
MINUTES = 20
MAX_GEN_HOURS = 4.0
SAMPLING = {"temperature": 0.8, "top_p": 0.95, "top_k": 20}   # a little more variety than the eval (0.6)
CU_PER_HOUR = 6.77
SEED = 3407

# Selection
TRAIN_MAX = 16384           # MAX_SEQ_LENGTH of 05
EASY_KEEP, MAX_PER_TASK = 0.5, 2

In [ ]:
import os, sys, json, glob, random, time, hashlib, importlib, collections, subprocess, shutil
from datetime import datetime
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")
login(token=userdata.get("HF_TOKEN"))
for f in ("claude_code.py", "web_mcp.py", "agent.py", "run_eval.py"):
    assert os.path.exists(f"{EVAL_DIR}/{f}"), f"{EVAL_DIR}/{f} not found: copy the repo's current eval/ folder to Drive"
sys.path.insert(0, EVAL_DIR)
sys.path_importer_cache.pop(EVAL_DIR, None)
importlib.invalidate_caches()
import run_eval as R
import agent as A
import claude_code as CC
assert CLAUDE_CODE_VERSION == CC.CLAUDE_CODE_VERSION, "the pinned Claude Code versions differ"

# Claude Code and the model's commands run as the unprivileged user "agent", in /srv (outside /content).
# /content (with Google Drive under it) is closed to that user, so the model cannot touch Drive.
CC.ensure_user("agent")
os.makedirs(WORK, exist_ok=True)
os.chmod("/srv", 0o755)
os.chmod(WORK, 0o755)
os.chmod("/content", 0o700)
probe = subprocess.run(["runuser", "-u", "agent", "--", "ls", "/content/drive"], capture_output=True)
assert probe.returncode != 0, "the agent user can read /content/drive: do not run the model's commands"
print("Drive is closed to the agent user.")

# The web tools (mcp__web__search, mcp__web__fetch), outside /content so that the agent user can run them
os.makedirs("/srv/cc_tools", exist_ok=True)
shutil.copy(f"{EVAL_DIR}/web_mcp.py", "/srv/cc_tools/web_mcp.py")
MCP_CONFIG = CC.web_mcp_config("/srv/cc_tools/web-mcp.json", "/srv/cc_tools/web_mcp.py")
subprocess.run(["chmod", "-R", "a+rX", "/srv/cc_tools"], check=True)


def uid(prefix, text):
    return f"{prefix}-{hashlib.md5(text.encode()).hexdigest()[:12]}"


held_out = lambda qid: int(hashlib.md5(qid.encode()).hexdigest()[:2], 16) < 8   # the eval split of 02 (~3%)
os.makedirs(OUT_DIR, exist_ok=True)

## KodCode tasks

- The same filter as the coding pools of `02` (instruct style, medium/hard, functions imported from `solution`).
- Not in the held-out split (where every eval set comes from), not in round 1, agent 1 or harness 1.
- Saved to Drive as `tasks.jsonl`, so a rerun uses the same pool.

In [ ]:
from datasets import load_dataset

TASKS = f"{OUT_DIR}/tasks.jsonl"
if os.path.exists(TASKS):
    tasks = [json.loads(l) for l in open(TASKS, encoding="utf-8")]
else:
    seen = set()
    for path in (ROUND1_GEN, AGENT1_EPISODES, HARNESS1_TASKS):
        if os.path.exists(path):
            seen |= {json.loads(l)["id"].removeprefix("py:") for l in open(path, encoding="utf-8") if l.strip()}
    tasks = []
    stream = load_dataset("KodCode/KodCode-V1", split="train", streaming=True).shuffle(seed=SEED + 4, buffer_size=20_000)
    for r in stream:
        qid = r["question_id"]
        if (r["style"] != "instruct" or r["gpt_difficulty"] not in ("medium", "hard") or not r["test_info"]
                or "from solution import" not in r["test"] or held_out(qid)
                or any(uid(p, qid) in seen for p in ("code", "agent", "harness"))):
            continue
        stub = A.make_stub(r["test_info"])
        if stub:
            tasks.append(CC.kodcode_task("cc", r, stub, "cc_kodcode"))
        if len(tasks) >= POOL:
            break
    with open(TASKS, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(t, ensure_ascii=False) + "\n" for t in tasks)

# Repository tasks (saved to Drive like the KodCode pool; the libraries are cloned again after a restart)
REPO_TASKS = f"{OUT_DIR}/repo_tasks.jsonl"
repos = CC.clone_repos(CC.TRAIN_REPOS, "/srv/repos")
!chmod -R a+rX /srv/repos
if os.path.exists(REPO_TASKS):
    repo_pool = [json.loads(l) for l in open(REPO_TASKS, encoding="utf-8")]
else:
    repo_pool = []
    for name, path in repos.items():
        repo_pool += CC.repo_tasks(name, path, REPO_TASKS_PER_REPO, "/content/repo_gen", seed=SEED,
                                   workers=os.cpu_count(), source="cc_repo_train")
    with open(REPO_TASKS, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(t, ensure_ascii=False) + "\n" for t in repo_pool)
tasks = tasks + repo_pool
random.Random(SEED).shuffle(tasks)   # both kinds early in the run, in case the time budget ends it
print(f"tasks: {len(tasks)}", collections.Counter(t["source"] for t in tasks))
print(repo_pool[0]["prompt"][:600] if repo_pool else tasks[0]["prompt"][:600])

## Model + generation (resumable)

Smoke test first: two tasks, one run each, into a scratch file. Then the whole pool; runs already on Drive are skipped.

In [ ]:
if "proxy" in globals():   # rerun of this cell: the old proxy and server free their ports first
    proxy.close()
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait()
model_path = CC.download_gguf(GGUF_REPO, GGUF_PATTERN)
print("model:", model_path)
server = CC.start_llama_server(LLAMA_SERVER, model_path, SLOTS, SLOT_CONTEXT, seed=SEED)
proxy = CC.Proxy(port=8100, sampling=SAMPLING, turn_tokens=TURN_TOKENS, seed=SEED)

for p in ("/content/smoke.jsonl", "/content/smoke_conv.jsonl"):
    if os.path.exists(p):
        os.remove(p)
smoke_tasks = [t for t in (next((t for t in tasks if not t.get("repo")), None),   # one of each kind
                            next((t for t in tasks if t.get("repo")), None)) if t]
CC.run_tasks(smoke_tasks, 1, proxy, "/content/smoke.jsonl", CLAUDE_BIN, work_root=WORK, concurrency=2,
             max_turns=MAX_TURNS, minutes=MINUTES, user="agent", conversations_path="/content/smoke_conv.jsonl", label="smoke",
             mcp_config=MCP_CONFIG, context_tokens=SLOT_CONTEXT, compact=False)
for l in open("/content/smoke_conv.jsonl", encoding="utf-8"):
    c = json.loads(l)
    replies = [m for m in c["messages"] if m["role"] == "assistant"]
    print(f"{c['id']}: ok={c['ok']} end={c['end'][:80]} | {len(replies)} replies, "
          f"{sum(bool(m.get('reasoning_content')) for m in replies)} with thinking, "
          f"{sum(len(m.get('tool_calls') or []) for m in replies)} tool calls, clean={CC.is_clean(c['messages'])}")
    tools = {t["function"]["name"] for t in c["tools"]}
    assert {"mcp__web__search", "mcp__web__fetch"} <= tools, f"the web tools are missing: {sorted(tools)}"

In [ ]:
t0 = time.time()
CC.run_tasks(tasks, SAMPLES, proxy, f"{OUT_DIR}/results.jsonl", CLAUDE_BIN, work_root=WORK, concurrency=SLOTS,
             max_turns=MAX_TURNS, minutes=MINUTES, hours=MAX_GEN_HOURS, user="agent",
             conversations_path=f"{OUT_DIR}/conversations.jsonl", label="gen", mcp_config=MCP_CONFIG,
             context_tokens=SLOT_CONTEXT, compact=False)
hours = (time.time() - t0) / 3600
print(f"Time: {hours:.2f} h ≈ {hours * CU_PER_HOUR:.1f} CU")
for src, st in CC.summarize(f"{OUT_DIR}/results.jsonl").items():
    print(f"{src}: solved {st['correct']}/{st['n']}, avg {st['turns'] / st['n']:.1f} turns, end reasons {dict(st['ends'])}")

## Selection → SFT data

- Runs that ended on their own (`success`), passed the restored tests and are clean (see the top).
- Rendered with the model's own chat template; at most `TRAIN_MAX` tokens; shortest first per task.
- Output: `claude_code_sft.jsonl` in the format of `agent_sft.jsonl` (tool call arguments as objects).

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(TOKENIZER)
results = [json.loads(l) for l in open(f"{OUT_DIR}/results.jsonl", encoding="utf-8")]
runs_per_task = collections.Counter(r["id"] for r in results)
solved_per_task = collections.Counter(r["id"] for r in results if r["ok"])

rng = random.Random(SEED)
stats = collections.Counter()
by_task = collections.defaultdict(list)
for l in open(f"{OUT_DIR}/conversations.jsonl", encoding="utf-8"):
    c = json.loads(l)
    stats["runs"] += 1
    if not (c["ok"] and c["end"] == "success") or c.get("compacted"):
        continue
    stats["solved"] += 1
    if not CC.is_clean(c["messages"]):
        continue
    stats["clean"] += 1
    messages = CC.for_template(c["messages"])
    n = len(tok(tok.apply_chat_template(messages, tools=c["tools"], tokenize=False), add_special_tokens=False).input_ids)
    stats["max_tokens_seen"] = max(stats["max_tokens_seen"], n)
    if n > TRAIN_MAX:
        continue
    stats["fitting"] += 1
    stats["fitting_repo" if c["id"].startswith("repo-") else "fitting_kodcode"] += 1
    by_task[c["id"]].append((n, {"messages": messages, "tools": c["tools"], "source": "claude_code", "n_tokens": n}))

examples = []
for tid, good in by_task.items():
    good.sort(key=lambda x: x[0])
    if solved_per_task[tid] == runs_per_task[tid]:
        if rng.random() > EASY_KEEP:
            continue
        good = good[:1]
    else:
        good = good[:MAX_PER_TASK]
    examples += [ex for _, ex in good]
rng.shuffle(examples)
with open(f"{OUT_DIR}/claude_code_sft.jsonl", "w", encoding="utf-8") as f:
    f.writelines(json.dumps(ex, ensure_ascii=False) + "\n" for ex in examples)

print(f"tasks run: {len(runs_per_task)}, solved at least once: {len(solved_per_task)}, runs solved: {sum(solved_per_task.values())}/{len(results)}")
print(f"runs: {stats['runs']}, solved and finished: {stats['solved']}, clean: {stats['clean']}, "
      f"fitting {TRAIN_MAX}: {stats['fitting']} ({stats['fitting_repo']} repository), kept: {len(examples)} "
      f"(longest clean run: {stats['max_tokens_seen']} tokens)")
if examples:
    print(f"avg tokens: {sum(e['n_tokens'] for e in examples) / len(examples):.0f} -> {OUT_DIR}/claude_code_sft.jsonl")

In [ ]:
# Check: a rendered example (Claude Code's tools, thinking, tool calls as XML)
if examples:
    print(tok.apply_chat_template(examples[0]["messages"], tools=examples[0]["tools"], tokenize=False)[-3000:])

## Finish: stop spending CU

In [ ]:
proxy.close()
server.terminate()
from google.colab import runtime
runtime.unassign()